# 自主課題の実装例
第1回を踏まえて，各自で課題を設定し，実験する．例えば，より長い系列を扱うことや，他の計算課題に取り組むことが考えられる．

このNotebookでは，埋め込み・補助タスク・lossを変更する方法を，加算の例で説明する．自分の課題に必要な節を参照する．「準備」と「データと学習の実行方法」の後は，どの節からでも実行できる．

| 工夫したいもの | 実装例 | 編集する場所 | 設定 |
| --- | --- | --- | --- |
| 数値の埋め込み | 数字の値を埋め込みに反映する | [src/embeddings.py](../src/embeddings.py)：`NumberEmbedding.forward` | [number.yaml](../configs/extensions/number.yaml) |
| 位置の埋め込み | Abacusで数字列内の位置を表す | [src/embeddings.py](../src/embeddings.py)：`AbacusEmbedding.position_ids`，`forward` | [abacus.yaml](../configs/extensions/abacus.yaml) |
| 補助タスク | 各桁の繰り上がりを予測する | [src/carry.py](../src/carry.py)：`carry_targets`，[src/extension_models.py](../src/extension_models.py)：`CarryDecoder` | [carry.yaml](../configs/extensions/carry.yaml) |
| 学習用loss | 正解系列の確率からlossを計算する | [src/losses.py](../src/losses.py)：`sequence_loss` | [sequence.yaml](../configs/extensions/sequence.yaml) |

Pythonファイルの`EXERCISE_BEGIN`から`EXERCISE_END`が編集箇所の目印．

## 準備
[環境設定](../environment-setup.md)に沿って教材をGoogle Driveに置く．次のパスは自分の教材フォルダに合わせる．コードを編集したら保存・同期してから学習コマンドを実行する．

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/project-research

In [ ]:
%%capture
%pip install calt-x

In [ ]:
import wandb
wandb.login()

## データと学習の実行方法
学習の入口は[src/train_extensions.py](../src/train_extensions.py)．基礎の`train.py`と同じIOPipeline・ModelPipeline・TrainerPipelineを使い，YAMLの`experiment`で埋め込みやモデルを選ぶ．

学習データの生成関数は[src/addition.py](../src/addition.py)の`sum_problem`，表記を決める関数は[src/extension_tasks.py](../src/extension_tasks.py)の`format_problem`．このNotebookの例では，両入力と正解を0埋めし，下位の桁から並べる．

In [ ]:
from src.extension_tasks import format_problem

format_problem(['59', '1'], '60')

[data.yaml](../configs/addition/data.yaml)の設定でデータを生成する．生成済みの場合は次のセルを省略できる．埋め込みやlossを変更して比較するときは，同じデータを使う．

In [ ]:
!python -m src.generate --config_path configs/addition/data.yaml

以下は変更前のモデルを学習する例．[baseline.yaml](../configs/extensions/baseline.yaml)の学習設定と，`train.wandb.project`・`group`・`name`，`train.output_dir`を確認する．

各節の例でも，使用するYAMLを開いて設定を確認する．別の実験を行うときは，`train.wandb.name`と`train.output_dir`を変更する．

In [ ]:
from omegaconf import OmegaConf

cfg = OmegaConf.load('configs/extensions/baseline.yaml')
print(OmegaConf.to_yaml(cfg))

In [ ]:
!bash scripts/extensions.sh configs/extensions/baseline.yaml

In [ ]:
from IPython.display import Image, display

output_dir = OmegaConf.load('configs/extensions/baseline.yaml').train.output_dir
display(Image(filename=f'{output_dir}/evaluation/length_heatmap.png'))
display(Image(filename=f'{output_dir}/evaluation/length_curve.png'))

学習後はW&Bでloss・完全一致率・token正解率と生成評価を確認する．

学習後に桁数別の評価を行い，ヒートマップと曲線を表示する．評価する桁数・問題数は[configs/extensions/evaluate.yaml](../configs/extensions/evaluate.yaml)で指定する．比較する実験では同じ評価設定を使う．

## 1. 数値の埋め込み
[src/embeddings.py](../src/embeddings.py)の`NumberEmbedding.forward`を読む．入力はtoken IDの行列`(B, L)`，出力は埋め込み`(B, L, D)`．`B`はバッチサイズ，`L`は系列長，`D`はベクトルの次元．

この例は，通常のtoken埋め込みに「数字の値 / 9 × 学習可能なベクトル」を加える．`values`は数字tokenを0〜1へ写す表で，数字以外は0になる．token IDと数字の値は別物である．

```python
token_vectors = self.token(input_ids)
numbers = self.values[input_ids].unsqueeze(-1)
return token_vectors + numbers * self.value_direction
```

参考文献：Siavash Golkar et al. (2023). “xVal: A Continuous Numerical Tokenization for Scientific Language Models.” [arXiv:2310.02989](https://arxiv.org/abs/2310.02989).

In [ ]:
import torch
from src.embeddings import NumberEmbedding

embedding = NumberEmbedding(vocab_size=11, d_model=8, digit_ids=list(range(10)))
ids = torch.tensor([[0, 1, 9, 10]])  # この例では10を演算記号のIDとする
print(embedding.values[ids])
print(embedding(ids).shape)

`NumberEmbedding.forward`を書き換えると，token IDから埋め込みを作る計算を変更できる．

[number.yaml](../configs/extensions/number.yaml)の`model.input_embedding_type: number`でこのクラスを選ぶ．

編集して保存した後，次のコマンドで学習する．

In [ ]:
!bash scripts/extensions.sh configs/extensions/number.yaml

In [ ]:
from IPython.display import Image, display

output_dir = OmegaConf.load('configs/extensions/number.yaml').train.output_dir
display(Image(filename=f'{output_dir}/evaluation/length_heatmap.png'))
display(Image(filename=f'{output_dir}/evaluation/length_curve.png'))

## 2. 桁位置の埋め込み：Abacus
[src/embeddings.py](../src/embeddings.py)の`AbacusEmbedding.position_ids`と`forward`を読む．数字の連続区間ごとに1, 2, 3, …と位置を付け，数字以外では0に戻す．学習時には数字の位置に共通のランダムoffsetを加え，評価時には加えない．

```python
positions = self.position_ids(input_ids)
return self.token(input_ids) + self.position(positions)
```

参考文献：Sean McLeish et al. (2024). “Transformers Can Do Arithmetic with the Right Embeddings.” [arXiv:2405.17399](https://arxiv.org/abs/2405.17399).

In [ ]:
import torch
from src.embeddings import AbacusEmbedding

embedding = AbacusEmbedding(vocab_size=12, d_model=8, digit_ids=list(range(10)))
embedding.eval()
ids = torch.tensor([[9, 5, 10, 1, 0, 11, 0, 6]])  # 59+01=60を各数の下位から並べる
print(embedding.position_ids(ids))
embedding.train()
print(embedding.position_ids(ids))

数字列内の位置の付け方は`position_ids`，位置ベクトルの加え方は`forward`を編集する．[abacus.yaml](../configs/extensions/abacus.yaml)では，`model.input_embedding_type: abacus`でこのクラスを選び，`model.use_positional_embedding: none`で通常の位置埋め込みを外す．

`model.input_embedding_kwargs`の`capacity`は位置埋め込み表の大きさ，`max_offset`は学習時のoffsetの選択肢数．最長数字列の桁数 + `max_offset`が`capacity`以下になるように設定する．

編集して保存した後，次のコマンドで学習する．

In [ ]:
!bash scripts/extensions.sh configs/extensions/abacus.yaml

In [ ]:
from IPython.display import Image, display

output_dir = OmegaConf.load('configs/extensions/abacus.yaml').train.output_dir
display(Image(filename=f'{output_dir}/evaluation/length_heatmap.png'))
display(Image(filename=f'{output_dir}/evaluation/length_curve.png'))

## 3. 補助タスク：繰り上がりの予測
補助タスクを追加するには，補助ラベル，予測する層，補助lossを用意する．ここでは，答えの各桁に加えて，その桁から次の桁へ繰り上がるかを予測する．繰り上がれば1，それ以外は0．

補助ラベルは[src/carry.py](../src/carry.py)の`carry_targets`で作る．引数は下位から並べた同じ長さの数字列．例えば59 + 1では，一の位の9 + 1で繰り上がり，十の位の5 + 0 + 1では繰り上がらない．

In [ ]:
from src.carry import carry_targets

print(carry_targets('95', '10'))
print(carry_targets('99', '10'))
print(carry_targets('21', '30'))

`CarryCollator`は標準collatorを継承し，バッチに`carry_labels`を追加する．これは今回の逆順・0埋め加算表記専用である．表記を変えるときは，入力の読み取りとラベルの位置も変更する．

補助ラベルは答えの一の位から並ぶ．最上位の追加桁・EOS・PADは`-100`とし，補助ロスに含めない．補助ラベルはlossの計算にだけ渡し，Transformerの入力には含めない．

In [ ]:
from calt.io import IOPipeline, UserCallableLoadPreprocessor
from src.carry import CarryCollator

pipeline = IOPipeline.from_config(cfg.data)
pipeline.dataset_load_preprocessor = UserCallableLoadPreprocessor(format_problem)
io = pipeline.build()
source, target = format_problem([59, 1], 60)
example = {'input': pipeline.preprocessor(source), 'target': pipeline.preprocessor(target)}
batch = CarryCollator(io['tokenizer'])([example])
print(batch['labels'])
print(batch['carry_labels'])

[src/extension_models.py](../src/extension_models.py)の`CarryDecoder`は，Transformerの隠れ状態から繰り上がりを予測する線形層を追加する．`hidden`は`(B, L, D)`，`carry_logits`は`(B, L)`．

```python
self.carry_head = nn.Linear(config.d_model, 1)
```

補助ロスは二値分類の交差エントロピー．sigmoidは`binary_cross_entropy_with_logits`に含まれる．

```python
carry_logits = self.carry_head(hidden).squeeze(-1)
mask = carry_labels != -100
aux_loss = F.binary_cross_entropy_with_logits(
    carry_logits[mask], carry_labels[mask].float(),
)
return main_loss + self.config.carry_weight * aux_loss
```

編集箇所は`carry_targets`，`carry_head`，`training_loss`．[carry.yaml](../configs/extensions/carry.yaml)の`carry_weight`で補助ロスの重みを指定する．

In [ ]:
!bash scripts/extensions.sh configs/extensions/carry.yaml

In [ ]:
from IPython.display import Image, display

output_dir = OmegaConf.load('configs/extensions/carry.yaml').train.output_dir
display(Image(filename=f'{output_dir}/evaluation/length_heatmap.png'))
display(Image(filename=f'{output_dir}/evaluation/length_curve.png'))

## 4. 学習用lossの変更
学習用lossは[src/losses.py](../src/losses.py)の`sequence_loss`を編集する．引数の`logits`は各位置・各tokenの予測スコア`(B, L, V)`，`labels`は正解token IDの行列`(B, L)`．`V`は語彙数．返り値は，バッチ全体のlossを表すスカラーのTensor．

この例では，通常のtoken単位の交差エントロピーに，正解系列の確率から計算した項を加える．正解tokenの確率を$p_t$とすると，

$$q=\prod_t p_t=\exp\left(-\sum_t(-\log p_t)\right),\qquad L=L_{\mathrm{CE}}+\lambda\,\mathrm{mean}_{\mathrm{batch}}(1-q^\gamma)$$

```python
losses = token_losses(logits, labels)  # 各tokenの交差エントロピー：(B, L)
ce = losses.sum() / (labels != -100).sum()
log_q = -losses.sum(dim=1)             # 各系列の対数確率：(B,)
reward = torch.exp(power * log_q)
return ce + strength * (1 - reward).mean()
```

`token_losses`はPADの位置（`labels == -100`）を0にする．`q`にはEOSを含む．正解の過去tokenを与えて計算する確率であり，生成した答えの完全一致率とは異なる．

次のセルで，予測確率を変えたときのlossを計算する．

In [ ]:
import torch
from src.losses import mean_token_loss, sequence_loss

# 2tokenからなる正解系列。正解tokenの確率を変えてlossを計算する。
labels = torch.tensor([[0, 0]])
for p in [0.5, 0.8, 0.95, 0.99]:
    logits = torch.tensor([[[p, 1-p], [p, 1-p]]]).log()
    ce = mean_token_loss(logits, labels)
    loss = sequence_loss(logits, labels, strength=1.0, power=2.0)
    print(f'p={p:.2f}, q={p*p:.4f}, CE={ce.item():.4f}, loss={loss.item():.4f}')

lossの集約方法や追加項は`sequence_loss`で変更する．[sequence.yaml](../configs/extensions/sequence.yaml)の`sequence_weight`が$\lambda$，`sequence_power`が$\gamma$に対応する．

[src/extension_models.py](../src/extension_models.py)の`SequenceLossDecoder`はモデルを継承し，`training_loss`だけを差し替える．

```python
def training_loss(self, hidden, logits, labels, carry_labels=None):
    return sequence_loss(logits, labels,
                         strength=self.config.sequence_weight,
                         power=self.config.sequence_power)
```

`experiment: sequence`を指定すると，`train_extensions.py`がこのモデルを選ぶ．

`compute_metrics`は評価値を計算する場所で，学習用lossの変更には使わない．勾配を計算できるよう，lossはPyTorchのTensor演算で計算し，返す前に`.item()`や`.detach()`を使わない．

編集して保存した後，次のコマンドで学習する．

In [ ]:
!bash scripts/extensions.sh configs/extensions/sequence.yaml

In [ ]:
from IPython.display import Image, display

output_dir = OmegaConf.load('configs/extensions/sequence.yaml').train.output_dir
display(Image(filename=f'{output_dir}/evaluation/length_heatmap.png'))
display(Image(filename=f'{output_dir}/evaluation/length_curve.png'))

## 手法を比較する
実行した手法の曲線を重ねて表示する．`config_paths`には評価済みの手法を指定する．

In [ ]:
from support.extension_comparison import plot_comparison

config_paths = {
    'baseline': 'configs/extensions/baseline.yaml',
    'number': 'configs/extensions/number.yaml',
    'abacus': 'configs/extensions/abacus.yaml',
    'carry': 'configs/extensions/carry.yaml',
    'sequence': 'configs/extensions/sequence.yaml',
}
comparison = plot_comparison(config_paths)
display(Image(filename=str(comparison)))

## 保存したモデルを再評価する
評価設定を変更した場合は，学習せずに評価だけを実行できる．次の例ではAbacusのモデルを評価する．

In [ ]:
!python -m src.evaluate_extensions --config_path configs/extensions/abacus.yaml

## 計算課題や設定を変更する
| 変更する条件 | 編集場所 |
| --- | --- |
| 学習する桁数・計算課題 | [src/addition.py](../src/addition.py)：`sum_problem` |
| 数の表記 | [src/extension_tasks.py](../src/extension_tasks.py)：`format_problem` |
| 学習回数・batch size・学習率 | 選んだYAMLの`train` |
| 語彙 | [configs/addition/lexer.yaml](../configs/addition/lexer.yaml) |
| 生成長 | 選んだYAMLの`generation_max_length`，評価設定の`max_length` |

別の計算課題を評価するときは，評価設定の`answer_function`も変更する（例：乗算は`operator:mul`）．生成関数を変更したらデータを作り直す．繰り上がりの例を他の計算課題に使う場合は，補助ラベルの定義も変える．学習後に表記関数やモデルを編集する場合は，評価に使えるよう学習時のコードと設定を残す．